# EMNIST FPGA Cloud Classification — Measurement Code Development

This notebook documents the adaptation of the **MNIST FPGA cloud accuracy implementation** to **EMNIST Balanced (47 classes)**, followed by the creation of a **continuous cloud-inference workload** for power measurements. The MNIST architecture and workflow were reused; this notebook records the changes rather than reproducing the complete MNIST procedure.

**Scope:** Vivado 2018.1 / Xilinx SDK 2018.1; ZedBoard Rev. E (Zynq-7020); embedded PS-to-Google Cloud Run classification. This is a **development/process record**, not the repository of formal result CSVs or cloud service deployment code.

## 1. Source implementations and code provenance

The starting point was the working MNIST cloud-assisted FPGA measurement workflow: stored image data in PL memory → shared BRAM mailbox → Zynq PS → Ethernet/TLS HTTPS request → remote prediction → prediction returned to PL. The EMNIST **local classification measurement** branch already contains the reusable image-ROM generation and dataset label resources; they were not copied again into this branch.

- Reused source: [`EMNIST-fpga-local-classification-measurements-code`](https://github.com/kjstacey/J00287507-Dissertation-AI-on-the-Edge/tree/EMNIST-fpga-local-classification-measurements-code) (link may need case adjustment to match the actual Git branch).
- Separate live-cloud source: `EMNIST-FPGA-Cloud-Live-Classification-Code`.
- This branch retains the **cloud measurement-specific** top-level hardware, SDK variants, programmable output, and block-design documentation.

The reusable OV7670 camera, VGA, clocking, and other unchanged hardware files remain in their original branches; their absence here does **not** mean they are unnecessary for rebuilding the full Vivado project.

## 2. Measurement-specific files in this branch

| Filename | Role |
|---|---|
| [`design_1_wrapper_speed.bit`](design_1_wrapper_speed.bit) | Uploaded programmed speed/measurement configuration |
| [`top_speed.vhd`](top_speed.vhd) | EMNIST speed/controlled-batch top-level logic |
| [`design_1_wrapper_continuous.bit`](design_1_wrapper_continuous.bit) | Programmed continuous-inference/power configuration |
| [`top_continuous.vhd`](top_continuous.vhd) | Continuous-workload top-level logic |
| [`design_1_bd.tcl`](design_1_bd.tcl) | Vivado IP Integrator block design reconstruction |
| [`mainemnistaccuracy.c`](mainemnistaccuracy.c) | SDK main program for controlled EMNIST accuracy measurements |
| [`tcp_clientemnistaccuracy.c`](tcp_clientemnistaccuracy.c) | Network client with EMNIST controlled-measurement processing |
| [`main_emnist_power.c`](main_emnist_power.c) | SDK main program for sustained inference/power workload |
| [`tcp_client_power.c`](tcp_client_power.c) | Network client for sustained inference/power workload |
| [`emnist_measurements_block_diagram.png`](emnist_measurements_block_diagram.png) | Vivado measurement architecture visual |

**Configuration boundary:** Both the speed and continuous configurations have their own bitstream and top-level VHDL files. Shared Vivado infrastructure is documented once through `design_1_bd.tcl`; camera and ROM support files remain referenced from earlier branches.

## 3. Reused MNIST cloud-accuracy structure

```text
EMNIST Balanced test images (ROM; reused from local measurement work)
    ↓
PL image controller and byte packing
    ↓
shared AXI BRAM mailbox (196 x 32-bit image words)
    ↓
Zynq PS / standalone SDK application
    ↓
lwIP → TCP → Mbed TLS → HTTPS POST /predict
    ↓
remote EMNIST CNN (47 classes)
    ↓
JSON prediction → PS writes mailbox → PL reads result
    ↓
controlled classification comparison / run sequencing
```

The common 28 × 28 input dimensions permitted retention of the same 784-byte transfer, AXI BRAM mailbox, network stack, and request/response mechanics established for MNIST. The network architecture was **not rebuilt around the larger class count**.

## 4. From MNIST cloud accuracy to EMNIST cloud accuracy

The adaptations were limited to dataset and output handling plus their associated test configuration:

1. **Test input and labels:** Replaced MNIST ROM data with the appropriate 100-image EMNIST Balanced block and associated labels. Reused the local EMNIST project's ROM-generation/source files instead of storing duplicate copies here.
2. **Classification range:** Expanded the accepted prediction values from MNIST's 0–9 to EMNIST Balanced's **0–46**, with a six-bit PL classification field where needed.
3. **Remote endpoint/model:** Directed the SDK's request to the EMNIST service; the server model and its deployment files live in a separate cloud-service branch.
4. **Controlled sequence:** Preserved the 100-image-per-programmed-block workflow, mailbox handshaking, expected-label comparisons, and measurement timing established for MNIST.
5. **Result verification:** Used the SDK results records to check image indices, expected versus predicted classes, full 100-image completion, and communication errors before accepting a formal run.

The ROM data and embedded software must refer to the **same block** (0–99, 100–199, …, 900–999) for valid expected-label comparisons.

### Shared mailbox interface retained

| Offset | Meaning |
|---|---|
| `0x000–0x30C` | 196 image words = 784 packed pixel bytes |
| `0x310` | Status: `0` idle, `1` image ready, `2` result ready |
| `0x314` | Image index |
| `0x318` | Returned classification (EMNIST class 0–46) |
| `0x31C` | Expected class in controlled testing |

The MNIST measurement workflow also used later mailbox offsets for counters and timing records. Those details were reused as applicable rather than redefined here. See the separate measurement notebook/data branch for the log format and statistical treatment.

## 5. Controlled accuracy and speed-measurement configuration

The controlled EMNIST implementation used **ten blocks of 100 test images**, each with known labels. After checking one validation run for each block, **three formal runs per block** were retained. The formal dataset therefore comprises **30 runs and 3,000 requests**, representing **1,000 distinct images repeated three times**.

The speed/accuracy variant used:

- `top_speed.vhd` and `design_1_wrapper_speed.bit` for the uploaded PL configuration;
- `mainemnistaccuracy.c` and `tcp_clientemnistaccuracy.c` as the corresponding SDK source variants;
- `design_1_bd.tcl` for the reusable Zynq/AXI BRAM block design.

The SDK measured elapsed client-observed request/response time over an already established TCP/TLS connection. Validation first-request delays and incomplete/error runs were not combined with accepted formal results. **These timings are not isolated cloud CNN inference timings.**

## 6. Adaptation to continuous cloud inference

After controlled accuracy and speed testing, the project was adapted using the existing MNIST sustained-workload/power-measurement process as a guide. The goal was to keep the embedded client issuing cloud requests repeatedly, rather than stopping after a single 100-image measurement sequence.

The continuous configuration is preserved as `top_continuous.vhd` and `design_1_wrapper_continuous.bit`, paired with the SDK variants `main_emnist_power.c` and `tcp_client_power.c`. The essential change was to **repeat the available stored image sequence and continue request/response processing**, keeping the cloud connection and embedded processing active for external power observations. The experiment used the ZedBoard's LED7 completion activity as evidence that successive 100-image cycles continued during measurement.

This workload was for **sustained embedded-system power observation**, not another independent accuracy sample. In particular, repeated classifications of the same stored images do not increase the number of distinct evaluated inputs.

The separate continuous bitstream and top-level source distinguish the sustained workload from the controlled speed/accuracy configuration.

## 7. Build and test sequence (process record)

1. Start from the existing MNIST FPGA cloud accuracy design and EMNIST local test-ROM assets.
2. Use the EMNIST block data and matching expected labels; verify image word order through the BRAM mailbox.
3. Update PL/PS prediction handling for EMNIST classes 0–46.
4. Build the controlled speed/accuracy hardware and SDK application; connect to the EMNIST Cloud Run endpoint.
5. Confirm all 100 image requests and expected/predicted label alignment before recording formal runs.
6. Repeat with each of the ten EMNIST 100-image blocks, using the documented validation and formal-run sequence.
7. Adapt the SDK for repeatable ongoing requests (`main_emnist_power.c`, `tcp_client_power.c`).
8. Validate continuous completion on hardware, then measure idle versus sustained cloud-assisted board power with development USB disconnected.
9. Capture Vivado utilization/timing/power reports for the relevant *speed*, *continuous*, and *live* hardware configurations in the data branch.

**Rebuild note:** This branch is an archive of changed code and build artifacts, not a complete standalone Vivado/SDK project. Original reusable IP, constraints, source dependencies, test-ROM data, and SDK/BSP infrastructure must be obtained from the referenced project branches and original tool environment.

## 8. Available block-design evidence

![EMNIST Cloud Measurement Block Design](emnist_measurements_block_diagram.png)

The PNG above records the Vivado architecture associated with the measurement configuration. The editable/reconstructible IP Integrator block design is maintained separately as [`design_1_bd.tcl`](design_1_bd.tcl).

## 9. Optional file inventory check

The following cell checks the **exact files listed in the branch screenshot**. It does not require Vivado or Xilinx SDK, and it does not attempt to run the embedded application.

In [ ]:
from pathlib import Path

expected = [
    "design_1_wrapper_speed.bit", "top_speed.vhd",
    "design_1_wrapper_continuous.bit", "top_continuous.vhd",
    "design_1_bd.tcl",
    "tcp_client_power.c", "tcp_clientemnistaccuracy.c",
    "main_emnist_power.c", "mainemnistaccuracy.c",
    "emnist_measurements_block_diagram.png",
]
for name in expected:
    file = Path(name)
    print(f"{'FOUND  ' if file.is_file() else 'MISSING'}  {name}")

## 10. Relationship to experimental results and live proof of concept

The following content is **intentionally stored in other branches**:

- **Cloud measurement Notebook and Data:** original SDK logs, formal/validation CSV records, accuracy and latency analyses, Vivado *speed/continuous/live* reports, and power photographs/videos.
- **EMNIST FPGA Cloud Live Classification Code:** final camera-driven live cloud hardware and ARM application.
- **EMNIST FPGA Cloud Live Classification Notebook and Data:** camera-to-cloud demonstration and troubleshooting evidence (including successful classification of the letter F).
- **Cloud service branch:** EMNIST model hosting, `/predict` service, and deployment artifacts.
- **EMNIST local measurement code:** reused EMNIST ROM/expected-label generation and unchanged shared hardware sources.

**Security boundary:** Development TLS configuration encrypted traffic but lacked certificate verification and a cryptographically secure entropy source. These limits were inherited from the MNIST embedded implementation and should not be interpreted as production-grade server authentication.

**Purpose of this notebook:** Document *what changed*, *how the MNIST process was reused*, and *how the ongoing workload was established*. Numerical experimental findings and full measurement evidence belong in the separate measurement-data notebook.